# Does the Kaiser window on the sinc transfer filter earn its keep?

The transfer filter enters the V-cycle in exactly two places, and **neither of
them is the fine-grid gradient step**:

```
fine level    z <- prox( z - A( E^H E (B z) - y~ ) )      <- no R, no P
coarse level  z_c <- prox( z_c + eta.pi - A( R E^H E P z_c  -  R y~ ) )
                                              ^^^^^^^^^^^     ^^^^^
correction    z  <- z + alpha( P (w_c - z_c) )
```

So "look at `E^H(Ex - y)` with and without the window" only means something if
it is the **coarse** residual `R E^H E P z_c - R y~`. That is what this notebook
compares. It is a more discriminating test than it first looks, for a reason
worth stating up front: **the kernel `h` appears twice** in that expression --
once in `P`, once in `R` -- so whatever `h` does to the spectrum gets squared.

Both arms use the same length (`L = 8 = 4 x factor`), the same exact-adjoint
machinery, and the same `sigma_c = sigma . ||h||_2` bookkeeping. The only
difference is the window.

In [ ]:
import math
import sys
import pathlib

import matplotlib.pyplot as plt
import torch

# repo root, whether this runs from notebooks/ or from the root
_root = pathlib.Path.cwd()
if not (_root / "operators").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))

from operators import FFT2D, Mask, Sense
from operators.resample import (Resample, kaiser_sinc1d, noise_scale, prolong,
                                restrict)

torch.manual_seed(0)
ARMS = ("sinc", "sinc_unwindowed")
LABEL = {"sinc": "Kaiser-windowed", "sinc_unwindowed": "rectangular (no window)"}
COLOR = {"sinc": "#1f77b4", "sinc_unwindowed": "#d62728"}
N, FACTOR, L = 128, 2, 8
print({a: f"L={L}  sigma_c/sigma={noise_scale(filter=a):.4f}" for a in ARMS})

## 1. The two kernels

A truncated sinc is the textbook Gibbs situation. Cutting it off with a
rectangular window buys a **sharper transition** and pays for it with **ripple**
-- and the ripple is in the passband, where the coarse grid is supposed to be
faithful.

In [ ]:
w = torch.linspace(0, math.pi, 2049, dtype=torch.float64)
n = torch.arange(L, dtype=torch.float64) - (L - 1) / 2

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, win in (("sinc", True), ("sinc_unwindowed", False)):
    h = kaiser_sinc1d(FACTOR, L, dtype=torch.float64, windowed=win)
    H = (h[None, :] * torch.exp(-1j * w[:, None] * n[None, :])).sum(1).abs()

    ax[0].stem(n.numpy(), h.numpy(), linefmt=COLOR[a], markerfmt="o",
               basefmt=" ", label=LABEL[a])
    ax[1].plot(w / math.pi, H, color=COLOR[a], label=LABEL[a])
    ax[2].plot(w / math.pi, H ** 2, color=COLOR[a], label=LABEL[a])

ax[0].set(title="kernel taps  h[n]", xlabel="tap")
ax[1].set(title=r"$|H(\omega)|$  (one application: R, or P)",
          xlabel=r"$\omega/\pi$", ylim=(0, 1.4))
ax[2].set(title=r"$|H(\omega)|^2$  (the coarse round-trip P R)",
          xlabel=r"$\omega/\pi$", ylim=(0, 1.8))
for k in (1, 2):
    ax[k].axvline(0.5, color="k", ls=":", lw=1)
    ax[k].axhline(1.0, color="k", ls="--", lw=0.8)
    ax[k].axvspan(0, 0.4, color="k", alpha=0.05)
    ax[k].legend(fontsize=8)
ax[0].legend(fontsize=8)
fig.suptitle("shaded = passband the coarse grid must reproduce;  dotted = coarse Nyquist",
             fontsize=9, y=1.02)
fig.tight_layout()

print(f"{'':22s}" + "".join(f"{f'{c:.2f}pi':>9s}" for c in [.1, .25, .4, .5, .6, .75]))
for a, win in (("sinc", True), ("sinc_unwindowed", False)):
    h = kaiser_sinc1d(FACTOR, L, dtype=torch.float64, windowed=win)
    H = (h[None, :] * torch.exp(-1j * w[:, None] * n[None, :])).sum(1).abs()
    idx = [(w - c * math.pi).abs().argmin() for c in [.1, .25, .4, .5, .6, .75]]
    print(f"{a + '  |H|':22s}" + "".join(f"{H[i]:9.3f}" for i in idx))
    print(f"{a + '  |H|^2':22s}" + "".join(f"{H[i]**2:9.3f}" for i in idx))

Read the middle table rows. The rectangular sinc is **better above** the coarse
Nyquist (0.136 vs 0.267 at `0.6pi` -- a sharper cutoff) and **worse below it**
(1.272 vs 0.957 at `0.25pi` -- 27% Gibbs overshoot).

Squared, that overshoot becomes **62% amplification**. Keep that number.

## 2. The test as asked: the coarse `E^H(E x - y)`

A 4x-undersampled multicoil SENSE operator, a phantom with sharp edges (Gibbs
needs an edge to ring off), and the coarse data residual
`R E^H E P z_c - R y~` under each arm.

In [ ]:
smaps = torch.randn(1, 4, N, N, dtype=torch.complex64)
smaps = smaps / (smaps.abs().pow(2).sum(1, keepdim=True).sqrt() + 1e-8)
mask = torch.zeros(1, 1, N, N)
mask[..., ::4, :] = 1                      # R = 4
mask[..., N // 2 - 8:N // 2 + 8, :] = 1    # + ACS
E = Mask(mask.to(torch.complex64)) @ FFT2D() @ Sense(smaps)

x_true = torch.zeros(1, 1, N, N)
x_true[..., 30:98, 30:98] = 1.0
x_true[..., 50:78, 50:78] = 0.4
x_true[..., 60:68, 20:108] = 0.8           # a thin bar: high-frequency content
x_true = (x_true + 0.02 * torch.randn(1, 1, N, N)).to(torch.complex64)

y = E(x_true)
y_tilde = E.adjoint(y)                     # what the network actually carries

resid = {}
for a in ARMS:
    z_c = restrict(x_true, filter=a)                 # a coarse iterate
    E_c = E @ Resample(FACTOR, None, a)              # E_c = E . P
    resid[a] = E_c.gram(z_c) - restrict(y_tilde, filter=a)

fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
vmax = max(r.abs().max().item() for r in resid.values())
ax[0].imshow(x_true.abs()[0, 0], cmap="gray"); ax[0].set_title("phantom (fine grid)")
for k, a in enumerate(ARMS):
    im = ax[k + 1].imshow(resid[a].abs()[0, 0], cmap="magma", vmin=0, vmax=vmax)
    ax[k + 1].set_title(f"|coarse residual|\n{LABEL[a]}")
    plt.colorbar(im, ax=ax[k + 1], fraction=0.046)
d = (resid[ARMS[0]] - resid[ARMS[1]]).abs()
im = ax[3].imshow(d[0, 0], cmap="magma", vmin=0, vmax=vmax)
ax[3].set_title("|difference|")
plt.colorbar(im, ax=ax[3], fraction=0.046)
for a_ in ax:
    a_.set_xticks([]); a_.set_yticks([])
fig.tight_layout()

n0 = resid["sinc"].abs().norm()
print(f"||r||  windowed      = {n0:.4f}")
print(f"||r||  unwindowed    = {resid['sinc_unwindowed'].abs().norm():.4f}")
print(f"||difference||       = {d.norm():.4f}   "
      f"({(d.norm() / n0).item() * 100:.0f}% of ||r_windowed||)")

So the answer to *"is this a sensible way to test it?"* is **yes, but only in
this coarse form, and only once you know what to look for.** The residual under
the rectangular window is roughly **3x larger in norm**. That is not a subtle
difference and it is not the undersampling artifact -- both arms see the
identical `E`.

Two caveats before reading anything into the pictures:

- The **fine-grid** `E^H(Ex - y)` would have shown nothing at all. The filter
  does not appear there.
- A single residual image conflates the two things the window trades off. The
  next cell separates them, which is what actually settles the question.

## 3. Why they differ: `P R` should be a projection, and one of them amplifies

The coarse level's job is to solve a *faithful smaller copy* of the fine
problem. The operator that says whether it is faithful is the round-trip
`P R`: on content the coarse grid can represent, `P R x` should return `x`.

`P R` applies `h` twice, so its passband gain is `|H|^2` -- and that is where
the 27% ripple becomes 62% amplification.

In [ ]:
fy = torch.fft.fftfreq(N)[:, None] * 2 * math.pi
fx = torch.fft.fftfreq(N)[None, :] * 2 * math.pi
rad = torch.sqrt(fy ** 2 + fx ** 2)
lowpass = rad <= math.pi / 2

def band(t, keep):
    return torch.fft.ifft2(torch.fft.fft2(t) * keep).real

noise = torch.randn(1, 1, N, N)
x_lo, x_hi = band(noise, lowpass), band(noise, ~lowpass)

rows = []
for a in ARMS:
    # (a) faithfulness: how far P R is from the identity on representable content
    pr = prolong(restrict(x_lo, filter=a), filter=a)
    faith = ((pr - x_lo).norm() / x_lo.norm()).item()
    # (b) amplification: gain of P R on a single mid-passband tone
    yy = torch.arange(N).float()
    tone = (torch.sin(2 * math.pi * 16 * yy / N)[None, None, :, None]
            * torch.ones(N)[None, None, None, :])
    gain = (prolong(restrict(tone, filter=a), filter=a).norm()
            / tone.norm()).item()
    # (c) aliasing: content the coarse grid CANNOT represent should be annihilated
    leak = (restrict(x_hi, filter=a).norm()
            / restrict(x_lo, filter=a).norm()).item()
    rows.append((a, faith, gain, leak, noise_scale(filter=a)))

print(f"{'arm':26s}{'||PRx-x||/||x||':>17s}{'P R gain @0.25pi':>18s}"
      f"{'alias leak':>13s}{'sigma_c/sigma':>15s}")
for a, f_, g, lk, ns in rows:
    print(f"{LABEL[a]:26s}{f_:>17.4f}{g:>18.4f}{lk:>13.4f}{ns:>15.4f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
names = [LABEL[a] for a, *_ in rows]
for k, (title, vals, good) in enumerate([
        (r"$\|PRx-x\|/\|x\|$   (lower better)", [r[1] for r in rows], "lower"),
        ("P R gain on a passband tone\n(1.0 = faithful)", [r[2] for r in rows], "one"),
        ("alias leak   (lower better)", [r[3] for r in rows], "lower")]):
    ax[k].bar(names, vals, color=[COLOR[a] for a, *_ in rows])
    ax[k].set_title(title, fontsize=10)
    ax[k].tick_params(axis="x", labelsize=8)
    if good == "one":
        ax[k].axhline(1.0, color="k", ls="--", lw=1)
fig.tight_layout()

This is the result, and it is **not** the story the window is usually sold with:

- On **aliasing** -- the thing an anti-alias filter exists for -- the
  rectangular sinc is slightly **better**. Its cutoff is sharper.
- On **faithfulness** it is twice as bad, because `P R` amplifies mid-passband
  content by ~60% instead of leaving it alone.

The window is not winning by suppressing aliasing. It is winning by **not
amplifying the signal the coarse grid is supposed to carry**. An inflated
coarse iterate produces an inflated coarse residual (section 2), and the FAS
correction then spends itself undoing an error the transfer operator invented.

It also passes 43% more noise: `sigma_c/sigma` is 0.597 unwindowed vs 0.416
windowed. `restrict_noise` tracks that automatically, so the coarse prox
thresholds follow -- but the coarse level genuinely is noisier.

## 4. Is any of this a fair test?

Sharper, per section 3: **the ranking depends on which failure you care about**,
and the residual picture in section 2 shows the combined effect rather than
either one alone. What this notebook establishes is a *mechanism* and a
*hypothesis*, not a result:

> The rectangular sinc inflates the coarse problem by ~60% in the passband; the
> Kaiser window trades a little extra aliasing to avoid that.

What it **cannot** establish is whether that matters end to end, for one
specific reason: `alpha` (the coarse-correction step), `eta` (the `pi` scale),
and every prox threshold are **learned**. A 1.6x systematic gain on the coarse
correction is exactly the kind of thing a learned scalar can absorb. The
network may simply shrink `alpha` and be no worse off.

So the honest chain is:

| question | settled by |
|---|---|
| what does the window change? | section 1, the kernel |
| does it reach the iteration? | section 2, the coarse residual (3x) |
| through what mechanism? | section 3, `P R` amplification |
| **does it change reconstruction quality?** | **`slurm/transfer_window.sbatch`** |

Run the last one before believing anything about which arm is better. The
diagnostic below is the thing worth logging during training -- if `alpha`
collapses in the unwindowed arm, that is the network absorbing the gain, and
the ablation will come out flat.

In [ ]:
from models.multigrid import MGCDLNet, VCycle

for a in ARMS:
    torch.manual_seed(0)
    m = MGCDLNet(K=[1, [2, 2, 2]], M=16, C=1, P=3, is_complex=False,
                 preproc="image", transfer_filter=a)
    vcs = [v for v in m.modules() if isinstance(v, VCycle)]
    alphas = [float(v.alpha.weight.detach().abs().mean()) for v in vcs]
    print(f"{LABEL[a]:24s} levels={len(vcs)}  "
          f"mean |alpha| at init = {[round(v, 5) for v in alphas]}")

print()
print("during training, log per level:")
print("    alpha.weight.abs().mean()      -- collapsing = absorbing the gain")
print("    vcycle.transfer.noise_scale    -- if learn_transfer=True")